# TiCoSS Demo

Run TiCoSS on one stereo image pair, visualize disparity and semantic segmentation, and optionally save the results.

In [ ]:
# Configure paths and runtime options here.
LEFT_IMAGE = "./demo_figures/left.png"
RIGHT_IMAGE = "./demo_figures/right.png"
CHECKPOINT = "./checkpoints/checkpoint.pth"

SAVE_RESULTS = True
SAVE_PATH = "./results"
OUTPUT_NAME = None  # None uses the left image file stem

VALID_ITERS = 32
DEVICE = "cuda"  # use "cpu" if needed
MIXED_PRECISION = False

In [ ]:
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from demo import (
    COLORMAP,
    autocast,
    load_model,
    read_image,
    visualize_disparity,
)
from core.utils.utils import InputPadder

for image_path in [LEFT_IMAGE, RIGHT_IMAGE, CHECKPOINT]:
    if not Path(image_path).exists():
        raise FileNotFoundError(f"File not found: {image_path}")

if DEVICE == "cuda" and not torch.cuda.is_available():
    print("CUDA is not available, falling back to CPU.")
    DEVICE = "cpu"

device = torch.device(DEVICE)
device

In [ ]:
# Model options mirror evaluate.py/demo.py defaults.
args = SimpleNamespace(
    restore_ckpt=CHECKPOINT,
    valid_iters=VALID_ITERS,
    device=DEVICE,
    mixed_precision=MIXED_PRECISION,
    is_train=False,
    hidden_dims=[128, 128, 128],
    corr_implementation="reg",
    shared_backbone=False,
    corr_levels=4,
    corr_radius=4,
    n_downsample=2,
    slow_fast_gru=False,
    n_gru_layers=3,
    scale=1,
    threshold=1.0,
    punishment=0.5,
)

model = load_model(args, device)

In [ ]:
@torch.no_grad()
def infer_pair(left_path, right_path):
    left_np, left_tensor = read_image(left_path)
    right_np, right_tensor = read_image(right_path)

    if left_np.shape[:2] != right_np.shape[:2]:
        raise ValueError(
            f"Left and right images must have the same size, got {left_np.shape[:2]} and {right_np.shape[:2]}."
        )

    image1 = left_tensor[None].to(device)
    image2 = right_tensor[None].to(device)

    padder = InputPadder(image1.shape, divis_by=32)
    image1, image2 = padder.pad(image1, image2)

    use_mixed_precision = args.mixed_precision or args.corr_implementation.endswith("_cuda")
    with autocast(enabled=use_mixed_precision):
        flow_pr, seg = model(
            image1,
            image2,
            image1,
            image2,
            threshold=args.threshold,
            iters=args.valid_iters,
            test_mode=True,
        )

    flow_pr = padder.unpad(flow_pr)
    seg = padder.unpad(seg)

    disparity = -flow_pr.squeeze(0).squeeze(0).detach().float().cpu().numpy()
    disparity_vis = visualize_disparity(disparity)

    pred_label = seg.argmax(1).squeeze(0).detach().cpu().numpy().astype(np.uint8)
    segmentation_vis = COLORMAP[pred_label]

    return {
        "left": left_np,
        "right": right_np,
        "disparity_vis": disparity_vis,
        "label": pred_label,
        "segmentation_vis": segmentation_vis,
    }

outputs = infer_pair(LEFT_IMAGE, RIGHT_IMAGE)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].imshow(outputs["left"])
axes[0, 0].set_title("Left")
axes[0, 1].imshow(outputs["right"])
axes[0, 1].set_title("Right")
axes[1, 0].imshow(outputs["disparity_vis"])
axes[1, 0].set_title("Disparity")
axes[1, 1].imshow(outputs["segmentation_vis"])
axes[1, 1].set_title("Segmentation")

for ax in axes.reshape(-1):
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
if SAVE_RESULTS:
    save_dir = Path(SAVE_PATH)
    save_dir.mkdir(parents=True, exist_ok=True)
    prefix = OUTPUT_NAME or Path(LEFT_IMAGE).stem

    Image.fromarray(outputs["disparity_vis"]).save(save_dir / f"{prefix}_disp.png")
    Image.fromarray(outputs["segmentation_vis"]).save(save_dir / f"{prefix}_seg.png")

    print(f"Saved visualizations to {save_dir.resolve()}")